# Difference order and controlled smoothness

The order $d$ defines which discrete derivative is penalized. For $d=1$, the infinite-penalty limit is constant (when $\mu=0$); for $d=2$, it is linear; for $d=3$, it is quadratic. These are theoretical limits of penalized least squares, not learned forecast models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother, difference_matrix

rng = np.random.default_rng(19)
t = np.linspace(-1, 1, 80)
y = 2 + 1.2 * t + 0.9 * t**2 + rng.normal(0, 0.25, t.size)

## Compare orders at equal normalized smoothness

Penalty values are **not directly comparable across orders**. Instead, ask each solver for the penalty that attains the same normalized smoothness level.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, y, '.', color='0.7', alpha=0.7, label='Synthetic observations')
for order in (1, 2, 3):
    solver = TrendSmoother(y.size, order=order)
    penalty = solver.lambda_for_smoothness(0.85)
    fit = solver.fit(y, lambda_=penalty, mu=0.0)
    ax.plot(t, fit.trend, label=f"order={order}, lambda={penalty:.3g}")
ax.set(xlabel='Normalized time', ylabel='Value', title='Same normalized smoothness, different orders')
ax.legend()
plt.show()

## Inspect the exact infinite-penalty limit

The library computes the nullspace projection directly at $\lambda=\infty$. This avoids choosing an arbitrarily large finite penalty and suffering avoidable numerical conditioning problems.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, y, '.', color='0.7', alpha=0.6, label='Observations')
for order in (1, 2, 3):
    solver = TrendSmoother(y.size, order=order)
    limit = solver.fit(y, lambda_=np.inf, mu=0.0)
    ax.plot(t, limit.trend, label=f"order={order}; EDF={limit.edf_conditional:.0f}")
ax.set(xlabel='Normalized time', ylabel='Value', title='Exact infinite-penalty trends')
ax.legend()
plt.show()

## Nonzero reference differences

Guerrero's formulation allows a nonzero reference mean $\mu$ for the differences. The mode `mu="data"` estimates this scalar from the observed differences; reported EDF remains **conditional** on that selected mean.

In [ ]:
solver = TrendSmoother(y.size, order=1)
fit = solver.fit(y, lambda_=20.0, mu="data")
print(f"Estimated observed-difference mean: {fit.mu:.4g}")
print(f"Difference-matrix shape: {difference_matrix(len(y), 1).shape}")